# **PART II: RAG SYSTEM**

Group Members:

  - Casucci Leonardo, student ID: 2196383
  - Frigo Gianmaria,  student ID: 2196376

Master Program: Computer Engineering

[Short Description and project goal]

In [1]:
from pathlib import Path

CACHE_DIR = Path("artifacts")
RAW_DATA_PATH = CACHE_DIR / "raw_dataset.parquet"
CHUNKS_DF_PATH = CACHE_DIR / "chunks_df.parquet"
EMBEDDINGS_PATH = CACHE_DIR / "embeddings.npy"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_DATA = FORCE_RECOMPUTE
FORCE_RECOMPUTE_CHUNKS = FORCE_RECOMPUTE
FORCE_RECOMPUTE_EMBEDDINGS = FORCE_RECOMPUTE

# **LIBRARIES USED**

This RAG system is built using a combination of specialized libraries for data processing, vector search, and LLM orchestration:

*   **Data Handling:** `pandas` and `numpy` for efficient manipulation of the email dataset and embedding arrays.
*   **Text Processing:** `re` for cleaning and `langchain_text_splitters` for intelligent, model-aware chunking.
*   **LLM & Embeddings:** `sentence-transformers` for generating high-quality vector embeddings, and `llama-cpp-python` for high-performance GGUF inference.
*   **Efficient Inference:** We utilize the **GGUF** format for Gemma-4-E4B, which allows for extremely efficient memory usage without the need for external runtime quantization libraries like `bitsandbytes`.
*   **Vector Store:** `faiss` (Facebook AI Similarity Search) for high-performance semantic retrieval.
*   **Evaluation:** `google-generativeai` to interface with Gemini 1.5 Pro for ground-truth generation and automated judging.

In [2]:
!pip install langchain-text-splitters

In [34]:
!pip install faiss-cpu

In [32]:
!pip install llama-cpp-python

In [4]:
import os
import re
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from getpass import getpass
import gc

# LLM & Chunking
from llama_cpp import Llama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

# Vector Search
import faiss

# Evaluation (Gemini API)
import google.generativeai as genai

# Device Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project initialized on device: {device}")

Project initialized on device: cuda


# **DATASET**

This project utilizes the **Jeffrey-Epstein-Emails-From-Epstein-Files** dataset, a collection of leaked and declassified communications recovered from various legal investigations.

#### Characteristics & Descriptive Statistics

The dataset consists of approximately **14,835 emails** after initial filtering. Key observations from the data distribution include:

*   **Highly Skewed Lengths:** The email bodies vary significantly, with a mean length of ~1,121 characters but a maximum reaching over **354,000 characters** (the equivalent of a small book). This extreme variation necessitated our specialized dynamic chunking strategy.
*   **Dense Communication Threads:** 75% of the emails are under 900 characters, indicating that a large portion of the dataset consists of short, direct exchanges that require precise chronological context for interpretation.
*   **Rich Metadata:** Each record contains critical investigative fields, including the original source document IDs, detailed sender/recipient names, and timestamps.
*   **Redaction Presence:** Many names and emails are redacted (e.g., `[redacted]`), but structural artifacts like angle brackets `<...>` remain, allowing us to still quantify the number of participants in group conversations.

## Dataset Loading

In [5]:
from datasets import load_dataset
import pandas as pd

if RAW_DATA_PATH.exists() and not FORCE_RECOMPUTE_DATA:
    df = pd.read_parquet(RAW_DATA_PATH)
else:
    dataset = load_dataset(
        "KillerShoaib/Jeffrey-Epstein-Emails-From-Epstein-Files"
    )
    df = dataset["train"].to_pandas()
    df.to_parquet(RAW_DATA_PATH, index=False)


df.head()

,doc_id,subject,preview,from_name,from_email,to,date,body
0,4accfb5f3ed84656e9762740081a4579,cody rudland: You are dead,- Lol good riddance,cody rudland,[redacted],<jeeproject@yahoo.com>,"Aug 13, 2019 6:48 AM",Lol good riddance
1,97d4a52d1df3948368770068262d2aab,Cecilia Steen: With love,"- My dearest Jeffrey, I don't know when you'l...",Cecilia Steen,[redacted],"JE project <jeeproject@yahoo.com>, Jeffrey Eps...","Jul 25, 2019 9:16 AM","My dearest Jeffrey,\nI don't know when you'll ..."
2,HOUSE_OVERSIGHT_016203,Flipboard 10 for Today: 11 questions for Mueller,- Flipboard Interesting stories worth your ti...,Flipboard 10 for Today,editorialstaff@flipboard.com,jeevacation@gmail.com,"Jul 13, 2019 9:06 PM",Flipboard\nInteresting stories worth your time...
3,HOUSE_OVERSIGHT_028801,"Flipboard Week in Review: Alex Acosta resigns,...",- Flipboard Biggest news stories from the pa...,Flipboard Week in Review,editorialstaff@flipboard.com,jeevacation@gmail.com,"Jul 13, 2019 6:48 PM",Flipboard\nBiggest news stories from the past ...
4,HOUSE_OVERSIGHT_029504,[redacted]: [redacted],"- Dear Jeff, I read the news papers today, I ...",[redacted],[redacted],Jeffrey Epstein <jeevacation@gmail.com>,"Jul 10, 2019 8:07 AM","Dear Jeff,\nI read the news papers today, I ca..."


## Dataset Profiling

Let's take a look at what our dataset looks like.
In particular we are intersted in:

    - number of entries
    - number of unique senders
    - number of unique recipients
    - average length of the answers

In [6]:
df.shape
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14835 entries, 0 to 14834
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   doc_id      14835 non-null  object
 1   subject     14835 non-null  object
 2   preview     14835 non-null  object
 3   from_name   14835 non-null  object
 4   from_email  14835 non-null  object
 5   to          14835 non-null  object
 6   date        14835 non-null  object
 7   body        14835 non-null  object
dtypes: object(8)
memory usage: 927.3+ KB


,0
doc_id,0
subject,0
preview,0
from_name,0
from_email,0
to,0
date,0
body,0


In [7]:
print(f"Number of unique senders: {df['from_email'].nunique()}")
print(f"Number of unique recipients: {df['to'].nunique()}")

Number of unique senders: 455
Number of unique recipients: 1434


In [8]:
df["body"].str.len().describe()

,body
count,14835.000000
mean,1121.424065
std,6707.694938
min,1.000000
25%,84.000000
50%,377.000000
75%,893.000000
max,354403.000000


In [9]:
df["subject"].value_counts().head(10)

,count
subject,
jeffrey E.: Re:,1685
J. Epstein: Re:,519
J: Re:,278
J. Epstein: (no subject),220
jeffrey E.: Re: Fwd:,217
jeffrey E.: (no subject),198
Jeffrey Epstein: Re:,173
[redacted]: Re:,172
Gmax: (no subject),142


The results of the profiling are clear: we have a considerable dataset (about 15000 entries) but entries are very short, in fact 50% of them are under 377 characters (usually 3/4 sentences). The 2 outliers at the extremes are notable: there's a 1 character entry and one of size 350000. The high skeweness of the dataset will probably make it difficult to accurately retreive information.

#### Dataset Clean Up
Checking for near-empty bodies.

In [10]:
df["body_len"] = df["body"].str.len()

short_bodies = df[df["body_len"] < 20]
df.sort_values("body_len")[["doc_id", "subject", "body", "body_len"]].head(20)

,doc_id,subject,body,body_len
1324,HOUSE_OVERSIGHT_032788,jeffrey E.: Re:,?,1
5894,HOUSE_OVERSIGHT_029831,[redacted]: Re: Reuters / lawsuit against Jeff...,j,1
1334,HOUSE_OVERSIGHT_032785,jeffrey E.: Re: [redacted],?,1
459,HOUSE_OVERSIGHT_028576,J: RE: Re:,?,1
501,HOUSE_OVERSIGHT_028589,J: Re:,?,1
2169,HOUSE_OVERSIGHT_032706,Kathy Ruemmler: Re:,?,1
603,HOUSE_OVERSIGHT_026329,J: Re:,?,1
568,HOUSE_OVERSIGHT_028611,J: Re:,?,1
6290,4c8ebdda8fbdccef86a17cee3e82562d,jeffrey E.: Re:,3,1
5261,HOUSE_OVERSIGHT_031498,"Thomas Jr., Landon: Re: Saudi money",no,2


Here all these entries are short answers to bigger emails. We can see a few question marks and simple answers like "no" or "ok". The RAG will use sliding window context so that it is able to link these small answers to the more significant chunks.

#### Normalizing whitespace.

Collapse multiple whitespaces, tabs or newlines into a single whitespace, critical for saving tokens and improving search accuracy (the embedding focuses on the meaning of the sentence rather than the formatting).

In [11]:
import re

def normalize_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for col in ["subject", "preview", "from_name", "from_email", "to", "date", "body"]:
    df[col] = df[col].apply(normalize_text)

#### Counting recipients

In [12]:
import re

def count_recipients_by_angle_brackets(to_field):
    if not isinstance(to_field, str) or to_field.strip() == "":
        return 0

    return max(1, len(re.findall(r"<[^<>]+>", to_field)))

df["num_recipients"] = df["to"].apply(count_recipients_by_angle_brackets)
df[["to", "num_recipients"]].sort_values("num_recipients", ascending=False).head(20)

,to,num_recipients
8437,valdemar iodice <valdemar@iodice.com.br>cc isa...,67
1617,"Alan Rogers <[redacted]>, Jeffrey Epstein <jee...",34
12509,"Allen West <[redacted]>, Rafael Bardaji <[reda...",31
6388,"Allen West <[redacted]>, Rafael Bardaji <[reda...",31
7016,"Glenn Young <shakescenes@aol.com>, gm2127 <gm2...",20
6695,"<sstrohaver@nlg.com>, <jboltax@nextjump.co>, <...",20
5748,Gary Savitzky <GarySavitzky@kamillagordon.oran...,20
6580,Gary Savitzky <rafael.prates@prismabrasil.com....,20
5519,"Schecter, Daniel (CC) <[redacted]>, Dunlavey, ...",17
8440,<JEEPROJECT@YAHOO.COM>cc <ehanna@nysgmail.com>...,16


#### Analyzing Recipient Scope

Counting the number of recipients per email is essential for several reasons:

    -   Distinguishing between a private one-on-one exchange and a mass-distributed newsletter or group thread.
    -   Ensuring that emails with identical bodies but different distribution lists (e.g., a private BCC vs a group CC) are not accidentally collapsed.
    -   Since many names in the 'to' field are redacted, counting the angle brackets '<...>' allows us to accurately estimate the "reach" of a message even when the participants' identities are hidden.

#### Row Deduplication.

Checking if dates are formatted in a consistent way before removing duplicates.

In [13]:
import pandas as pd

df["parsed_date"] = pd.to_datetime(df["date"], errors="coerce")

print("Unparseable dates:", df["parsed_date"].isna().sum())
print("Date range:", df["parsed_date"].min(), "to", df["parsed_date"].max())

Unparseable dates: 0
Date range: 1970-01-01 00:00:00 to 2019-08-13 06:48:00


There is a large number of repeated records appears when using a content-based duplicate key

In [14]:
dedup_key = ["subject", "from_email", "to", "num_recipients", "date", "body"]

print("Full-row duplicates:", df.duplicated().sum())
print("Content-based duplicates:", df.duplicated(subset=dedup_key).sum())

Full-row duplicates: 11
Content-based duplicates: 1236


In [15]:
duplicate_groups = (
    df[df.duplicated(subset=dedup_key, keep=False)]
    .groupby(dedup_key)
    .agg(
        n_rows=("doc_id", "size"),
        doc_ids=("doc_id", lambda x: list(x)),
        n_unique_doc_ids=("doc_id", "nunique"),
        n_unique_previews=("preview", "nunique"),
        n_unique_from_names=("from_name", "nunique")
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

duplicate_groups.head(10)

,subject,from_email,to,num_recipients,date,body,n_rows,doc_ids,n_unique_doc_ids,n_unique_previews,n_unique_from_names
594,jeffrey E.: Re:,jeevacation@gmail.com,[redacted],1,"Sep 8, 2017 11:52 AM",what is your schedule in new york how long wil...,11,"[HOUSE_OVERSIGHT_032622, HOUSE_OVERSIGHT_02366...",11,10,1
595,jeffrey E.: Re:,jeevacation@gmail.com,[redacted],1,"Sep 8, 2017 1:33 PM",I will get larson to call you,11,"[HOUSE_OVERSIGHT_032622, HOUSE_OVERSIGHT_02366...",11,10,1
319,"Thomas Jr., Landon: Re: Saudi money",[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Oct 19, 2016 9:41 AM",Interesting. CEO of big finance form told me t...,9,"[HOUSE_OVERSIGHT_031511, HOUSE_OVERSIGHT_03163...",9,9,1
621,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 8:01 AM","Yes, trains exist -- people ride them to get f...",8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
556,jeffrey E.: Re:,jeevacation@gmail.com,Nicholas Ribis,1,"May 27, 2017 6:20 PM",are you on cell?,8,"[HOUSE_OVERSIGHT_026123, HOUSE_OVERSIGHT_02611...",8,8,1
612,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 3:44 PM",Gov is not jewish,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
610,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 2:59 PM",Yep. And now he is a repeat offender.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
609,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 2:57 PM",Reid's guy went down on all 7 counts.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
619,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 7:15 AM",Most girls do not have to worry about this crap.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
611,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 3:02 PM","Uh, yeah. We will now go through a period of s...",8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1


For the RAG system, repeated identical email content is not useful and may cause the top-k results to contain multiple copies of the same evidence.

The internal document IDs are still useful for source reporting. Therefore, we collapse rows with identical email content into a single row that preserves all `doc_id`s.


In [16]:
df_grouped = (
    df.groupby(dedup_key, dropna=False)
    .agg(
        doc_ids=("doc_id", lambda x: sorted(set(x))),
        n_original_rows=("doc_id", "size"),
        preview=("preview", "first"),
        from_name=("from_name", "first"),
        parsed_date=("parsed_date", "first")
    )
    .reset_index()
)

print("Original rows:", len(df))
print("Rows after content-based grouping:", len(df_grouped))
print("Rows collapsed:", len(df) - len(df_grouped))

Original rows: 14835
Rows after content-based grouping: 13599
Rows collapsed: 1236


#### Chunking
This function creates a structured header for each chunk, ensuring that metadata (Subject, Sender, Date, and Source IDs) is preserved and searchable during retrieval.

In [17]:
def build_header(row):
    # Show up to 5 document IDs to avoid long headers
    doc_ids = ", ".join(row["doc_ids"][:5])

    if len(row["doc_ids"]) > 5:
        doc_ids += f", ... ({len(row['doc_ids'])} total)"

    # Truncate recipient list if it's too long
    recipients = str(row["to"])
    if len(recipients) > 200:
        recipients = recipients[:200].rstrip() + " ... (truncated)"

    # Truncate subject if it's too long
    subject = str(row["subject"])
    if len(subject) > 200:
        subject = subject[:200].rstrip() + " ... (truncated)"

    return f"""Subject: {subject}
From: {row['from_name']} <{row['from_email']}>
To: {recipients} of {row['num_recipients']} total recipients
Date: {row['date']}
Source document IDs: {doc_ids}
"""

#### Tokenizer and Splitter Setup

Initializing the model-specific tokenizer and the base recursive splitter used for dividing the email body into manageable segments.

In [18]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""]
)

#### Dynamic Token-Aware Chunking

We use a dynamic chunking strategy that calculates the header's token length for each email and adjusts the body's `chunk_size` accordingly. This guarantees that the combined text (Header + Body) consistently fits within the 512-token limit of the embedding model.

In [19]:
if CHUNKS_DF_PATH.exists() and not FORCE_RECOMPUTE_CHUNKS:
    chunks_df = pd.read_parquet(CHUNKS_DF_PATH)
else:
    chunk_records = []

    for row_idx, row in df_grouped.iterrows():
        header = build_header(row)

        # Calculate tokens in header to adjust body chunk size
        # We use a safety margin of 15 tokens for special tokens and chunk info text
        header_tokens = len(tokenizer.encode(header, add_special_tokens=False))
        max_body_tokens = 512 - header_tokens - 15

        # Ensure we don't have a negative or tiny chunk size
        chunk_size = max(128, max_body_tokens)

        local_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=tokenizer,
            chunk_size=chunk_size,
            chunk_overlap=int(chunk_size * 0.15),
            separators=["\n\n", "\n", ". ", " ", ""]
        )

        body_chunks = local_splitter.split_text(row["body"])

        for chunk_idx, body_chunk in enumerate(body_chunks):
            chunk_text = f"""{header}

Body chunk {chunk_idx + 1} of {len(body_chunks)}:
{body_chunk}"""

            chunk_records.append({
                "chunk_id": f"{row_idx}_{chunk_idx}",
                "email_id": row_idx,
                "chunk_index": chunk_idx,
                "num_chunks": len(body_chunks),

                # Metadata
                "doc_ids": row["doc_ids"],
                "n_original_rows": row["n_original_rows"],
                "subject": row["subject"],
                "from_name": row["from_name"],
                "from_email": row["from_email"],
                "to": row["to"],
                "date": row["date"],

                # Text used for retrieval and prompt
                "text_for_embedding": chunk_text,
                "text_for_prompt": chunk_text,
            })

    chunks_df = pd.DataFrame(chunk_records)
    chunks_df.to_parquet(CHUNKS_DF_PATH, index=False)

chunks_df.head()

Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


,chunk_id,email_id,chunk_index,num_chunks,doc_ids,n_original_rows,subject,from_name,from_email,to,date,text_for_embedding,text_for_prompt
0,0_0,0,0,8,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...
1,0_1,0,1,8,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...
2,0_2,0,2,8,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...
3,0_3,0,3,8,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...
4,0_4,0,4,8,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...


In [20]:
print(f"Description of number of chunks per email:\n{chunks_df['num_chunks'].describe()}")


Description of number of chunks per email:
count    23385.000000
mean       204.456062
std        395.742909
min          1.000000
25%          1.000000
50%          2.000000
75%          9.000000
max       1062.000000
Name: num_chunks, dtype: float64


In [21]:
chunks_df[chunks_df["text_for_embedding"].str.len() < 5]

,chunk_id,email_id,chunk_index,num_chunks,doc_ids,n_original_rows,subject,from_name,from_email,to,date,text_for_embedding,text_for_prompt


In [22]:
chunks_len = chunks_df["text_for_embedding"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)
longest = chunks_len.max()
longest_chunk = chunks_df.iloc[chunks_len.idxmax()]


print(f"Longest chunk with header: {longest} tokens")
print(f"Text of longest chunk:\n{longest_chunk['text_for_prompt']}")

Longest chunk with header: 507 tokens
Text of longest chunk:
Subject: : Fw: Emailing: 667 air ghislaine 008, 667 air ghislaine 009, 667 air ghislaine 010, 667 air ghislaine 011, 667 air ghislaine 006, 667 air ghislaine 007
From:  <lvjet@aol.com>
To: Jeffrey Epstein <jeeproject@yahoo.com> of 1 total recipients
Date: Mar 5, 2008 8:07 PM
Source document IDs: 94f9ed9bac6267a20a85b9a1b82277fb


Body chunk 397 of 1062:
MSIR@9X!/L*!#<Q 8"R.?<X%.$F/NQHOOUIX1NZA1[G%2*J#[SY/HHS3L!"7D M;J3CTI &/0&KBF+M&3_O'_"I$GV\!57Z"BPKE-+:=^1&?QXIQLW_(Y$7Z<UM:DE#=14)"'J:5AW(S;Q*/OLQ[YXH"J/N!![XR?UJ>'S3A!D^PS5F.PY>SQM1_4Y/Z4#,]@Q7YG)'UIJH.<J01Q@]:VEMM/C!\VXE=AT"@I,;>(MY"@9.2M3R?UI 5UA+?=0_6GB #[SOXY-#2Y[DTG)[4P) D(Z!V/OP*<IP/E"K]!S21MQ2N..U2&V<<O+&H]VI7 ;Y;,22Q/UI#"?6K<-EN0,9U/L :D-KLZ-[TK@9W MV:1B0O/TI&LW!^>5%^O-6Y591]X_2JKD>E,+C1!&I[O^E<)K49BU.X4\?/D?


# **EMBEDDING INDEX AND RETRIEVAL**

#### Embedding Generation Logic

This function transforms the processed text chunks into high-dimensional vectors (embeddings) using the SentenceTransformer model. These vectors represent the semantic meaning of the text.

In [23]:
def compute_embeddings(chunks_df, embedding_model, batch_size=64):

    texts = chunks_df["text_for_embedding"].tolist()

    embeddings = embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    return embeddings.astype("float32")

#### Embedding Caching System

To save time and computational resources, we implement a caching mechanism that stores the generated embeddings on disk. The system automatically detects if the saved embeddings are compatible with the current dataset before loading them.

In [24]:
def load_or_compute_embeddings(
    chunks_df,
    embedding_model,
    embeddings_path=EMBEDDINGS_PATH,
    force_recompute=False
):
    """
    Load embeddings from disk if available and compatible.
    Otherwise compute and save them.
    """
    if embeddings_path.exists() and not force_recompute:
        print(f"Loading embeddings from {embeddings_path}...")
        embeddings = np.load(embeddings_path)

        if embeddings.shape[0] == len(chunks_df):
            return embeddings.astype("float32")

        print(
            "Saved embeddings are incompatible with chunks_df "
            f"({embeddings.shape[0]} embeddings vs {len(chunks_df)} chunks). "
            "Recomputing..."
        )

    else:
        print("No saved embeddings found. Computing embeddings...")

    embeddings = compute_embeddings(chunks_df, embedding_model)

    np.save(embeddings_path, embeddings)
    print(f"Saved embeddings to {embeddings_path}")

    return embeddings

#### Model Loading and Execution

We use the `BAAI/bge-base-en-v1.5` model, which is one of the top-performing open-source models for retrieval tasks. This block initializes the model on the available device (GPU/CPU) and triggers the embedding process.

In [25]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=device
)

embeddings = load_or_compute_embeddings(
    chunks_df=chunks_df,
    embedding_model=embedding_model,
    force_recompute=FORCE_RECOMPUTE_EMBEDDINGS
)

embeddings.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

No saved embeddings found. Computing embeddings...


Batches:   0%|          | 0/366 [00:00<?, ?it/s]

Saved embeddings to artifacts/embeddings.npy


(23385, 768)

## Embedding Index Creation

#### Vector Index Construction (FAISS)

We use **FAISS** (Facebook AI Similarity Search) to build an efficient index of our embeddings. Specifically, `IndexFlatIP` (Inner Product) is used for high-speed cosine similarity search, allowing us to find relevant chunks in milliseconds even across thousands of documents.

In [26]:
import faiss

embedding_matrix = embeddings.astype("float32")

index = faiss.IndexFlatIP(embedding_matrix.shape[1])
index.add(embedding_matrix)

## Retrieval

#### Semantic Retrieval with Chronological Expansion

This is the core of our retrieval engine. Beyond finding the most semantically relevant chunks, this function also performs **Temporal Expansion** by retrieving the emails immediately preceding and following a hit. This provides critical conversation context to the LLM, especially for short messages like "Ok" or "?".

In [27]:
def retrieve(query, k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = chunks_df.iloc[indices[0]].copy()
    results["score"] = scores[0]

    return results

In [28]:
retrieve("What emails mention Bill Clinton private orgy?", k=5)

,chunk_id,email_id,chunk_index,num_chunks,doc_ids,n_original_rows,subject,from_name,from_email,to,date,text_for_embedding,text_for_prompt,score
20944,11428_0,11428,0,1,[HOUSE_OVERSIGHT_026047],1,jeffrey E.: RE: Re:,jeffrey E.,jeevacation@gmail.com,"Weingarten, Reid","Dec 20, 2016 12:05 PM",Subject: jeffrey E.: RE: Re:\nFrom: jeffrey E....,Subject: jeffrey E.: RE: Re:\nFrom: jeffrey E....,0.746318
19060,9931_0,9931,0,1,[HOUSE_OVERSIGHT_032855],1,Tom Barrack Private: (no subject),jeffrey E.,jeevacation@gmail.com,Tom Barrack Private <[redacted]>,"Mar 9, 2016 6:26 PM",Subject: Tom Barrack Private: (no subject)\nFr...,Subject: Tom Barrack Private: (no subject)\nFr...,0.733305
19457,10129_2,10129,2,10,[HOUSE_OVERSIGHT_022673],1,[redacted]: Fwd:,[redacted],[redacted],[redacted]cc [redacted],"Jan 30, 2015 12:25 PM",Subject: [redacted]: Fwd:\nFrom: [redacted] <[...,Subject: [redacted]: Fwd:\nFrom: [redacted] <[...,0.731071
19585,10240_0,10240,0,1,[HOUSE_OVERSIGHT_021806],1,[redacted]: Re:,jeffrey E.,jeevacation@gmail.com,[redacted],"Jan 3, 2015 5:21 PM",Subject: [redacted]: Re:\nFrom: jeffrey E. <je...,Subject: [redacted]: Re:\nFrom: jeffrey E. <je...,0.730459
19058,9929_0,9929,0,1,[HOUSE_OVERSIGHT_032243],1,Tom Barrack Private: (no subject),jeffrey E.,jeevacation@gmail.com,,"Mar 9, 2016 6:26 PM",Subject: Tom Barrack Private: (no subject)\nFr...,Subject: Tom Barrack Private: (no subject)\nFr...,0.725037


# **PROMPT ENGINEERING**

Prompt engineering is the final bridge between our retrieval engine and the LLM. For this project, we target **Gemma-2 (quantized)**, which requires a specific conversational format to function correctly.

#### Design Choices & Strategy

*   **Model-Specific Formatting:** We use the `<start_of_turn>user` and `<start_of_turn>model` tags to respect Gemma's instruction-tuning template.
*   **Role Definition:** We explicitly cast the LLM as an "investigative assistant" to set a professional and analytical tone.
*   **Strict Grounding:** To prevent hallucinations, the prompt includes a hard constraint: the model must only use the provided segments and must explicitly state if information is missing.
*   **Automatic Citations:** By prepending metadata headers to every chunk in the previous steps, we enable the model to cite "Source document IDs" for every fact it generates, ensuring full traceability back to the original Epstein files.

In [29]:
def build_prompt(query, k=5):
    """
    Retrieves relevant documents and crafts a structured prompt for the Gemma LLM.
    """
    retrieved_df = retrieve(query, k=k)

    # extract and format the context from retrieved documents
    context_parts = []
    for idx, (_, row) in enumerate(retrieved_df.iterrows()):
        context_parts.append(f"[Segment {idx+1}]\n{row['text_for_prompt']}")

    context_str = "\n\n".join(context_parts)

    # gemma promt template
    # Template: <start_of_turn>user\n{content}<end_of_turn>\n<start_of_turn>model\n
    system_instruction = (
        "You are an investigative assistant analyzing a dataset of emails from the Epstein files. "
        "Your task is to answer the user's question accurately using ONLY the provided email segments. "
        "For each piece of information you provide, you MUST cite the 'Source document IDs' found in the segment headers. "
        "If the provided context does not contain the answer, explicitly state that the information is not available in the retrieved data."
        "Answer only relevant questions based on the provided context. Do not speculate or provide information that is not supported by the retrieved email segments."
    )

    user_content = f"""{system_instruction}

    RELEVANT EMAIL SEGMENTS:
    {context_str}

    USER QUESTION: {query}"""

    prompt = f"<start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

    return prompt

# Example test
example_prompt = build_prompt("What was discussed regarding Saudi money?")
print(f"Generated prompt length: {len(example_prompt)} characters")
print("--- Prompt Preview ---")
print(example_prompt[:1000] + "...")

Generated prompt length: 3589 characters
--- Prompt Preview ---
<start_of_turn>user
You are an investigative assistant analyzing a dataset of emails from the Epstein files. Your task is to answer the user's question accurately using ONLY the provided email segments. For each piece of information you provide, you MUST cite the 'Source document IDs' found in the segment headers. If the provided context does not contain the answer, explicitly state that the information is not available in the retrieved data.Answer only relevant questions based on the provided context. Do not speculate or provide information that is not supported by the retrieved email segments.

    RELEVANT EMAIL SEGMENTS:
    [Segment 1]
Subject: Thomas Jr., Landon: Re: Saudi money
From: Thomas Jr., Landon <[redacted]>
To: jeffrey E. <jeevacation@gmail.com> of 1 total recipients
Date: Oct 19, 2016 1:41 PM
Source document IDs: HOUSE_OVERSIGHT_031496


Body chunk 1 of 1:
Importance: High Interesting. CEO of big finance fi

# **MODEL LOADING**

To run the advanced **Gemma-4-E4B** model efficiently, we utilize the **GGUF** format. Unlike standard PyTorch models, GGUF files are **pre-quantized**, meaning the weights are already optimized for low-memory environments (like Google Colab) during the file creation process. 

Consequently, we no longer need runtime quantization libraries like `bitsandbytes`. This significantly simplifies the loading process and ensures stable performance on a wide range of hardware.

In [33]:
import gc

# --- Memory Management ---
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

clear_memory()

# --- Authentication Setup ---
def get_hf_token():
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token: return token
    except: pass
    return input("Enter Hugging Face Token (Required for gated Gemma access): ").strip()

HF_TOKEN = get_hf_token()

# --- Model Configuration ---
model_id = "google/gemma-4-e4b-it-heretic-GGUF"
filename = "gemma-4-e4b-it-heretic.Q4_K_M.gguf"

print(f"Loading {model_id} via llama-cpp...")

llm = Llama.from_pretrained(
    repo_id=model_id,
    filename=filename,
    n_ctx=8192,
    n_gpu_layers=-1,
    token=HF_TOKEN
)

Loading google/gemma-2-9b-it...


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

# **GENERATION**

With the retrieval engine ready and the model loaded, we can now define the final function that orchestrates the RAG pipeline.

In [ ]:
def answer_question(query, k=5, max_new_tokens=1024):
    # 1. Build the contextual prompt
    prompt = build_prompt(query, k=k)
    
    # 2. Generate the response using llama-cpp-python
    # GGUF models via llama-cpp handle their own formatting internally
    # based on the prompt template provided in build_prompt.
    response = llm(
        prompt,
        max_tokens=max_new_tokens,
        echo=False,        # Don't include the prompt in the output
        temperature=0.1,
        top_p=0.95,
        stop=["<end_of_turn>", "<|endoftext|>"]  # Ensure it stops correctly
    )
    
    # 3. Extract the text
    answer = response["choices"][0]["text"].strip()
    
    return answer

# Example of a real RAG query
query = "What are the main discussions regarding the Saudi contract and who was involved?"
print(f"QUERY: {query}\n")
response = answer_question(query)
print(f"ANSWER:\n{response}")

# **TESTING AND EVALUATION**

To evaluate the system, we implement a three-stage automated pipeline using the Google AI Studio API (Gemini 1.5 Pro) as our "Ground Truth Generator" and "Expert Judge."

###  Setup Gemini API
You will need a free API key from [Google AI Studio](https://aistudio.google.com/).

In [ ]:
# Paths for caching evaluation results
TEST_SET_PATH = CACHE_DIR / "micro_test_set.json"
EVAL_RESULTS_PATH = CACHE_DIR / "evaluation_results.json"

def setup_gemini():
    # 1. Try to get from Colab Secrets
    try:
        from google.colab import userdata
        key = userdata.get('GOOGLE_API_KEY')
        if key: 
            os.environ["GOOGLE_API_KEY"] = key
            genai.configure(api_key=key)
            return genai.GenerativeModel('gemini-1.5-pro')
    except:
        pass
    
    # 2. Try to get from Environment or Manual Input
    if "GOOGLE_API_KEY" not in os.environ:
        key = input("Enter Google API Key (or press Enter to load pre-computed results from disk): ").strip()
        if key:
            os.environ["GOOGLE_API_KEY"] = key
        else:
            print("No API key provided. System will attempt to load results from artifacts/ folder.")
            return None

    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    return genai.GenerativeModel('gemini-1.5-pro')

gemini_model = setup_gemini()

### Generate Micro-Test Set (Ground Truth)
We randomly sample 15 emails and ask Gemini to generate complex question/answer pairs based on them.

In [ ]:
def generate_ground_truth(df, n=15):
    # Check for cache first
    if TEST_SET_PATH.exists():
        print(f"Loading test set from {TEST_SET_PATH}...")
        with open(TEST_SET_PATH, 'r') as f:
            return json.load(f)

    if not gemini_model:
        print("No API key and no cached test set found.")
        return []

    print("Generating new test set via Gemini API...")
    sample_df = df[df["body"].str.len() > 200].sample(n)
    test_set = []

    for _, row in sample_df.iterrows():
        context = build_header(row) + "\n\nBody: " + row["body"]
        prompt = f"Based on the following email passage, generate one complex question and its correct answer. Return JSON with keys 'question' and 'answer'.\n\nEMAIL PASSAGE:\n{context}"

        try:
            response = gemini_model.generate_content(prompt)
            json_str = response.text.strip().replace('```json', '').replace('```', '')
            qa = json.loads(json_str)
            qa["source_doc_ids"] = row["doc_ids"]
            test_set.append(qa)
        except:
            continue

    # Save cache
    with open(TEST_SET_PATH, 'w') as f:
        json.dump(test_set, f)
    return test_set

micro_test_set = generate_ground_truth(df_grouped)
print(f"Test set ready with {len(micro_test_set)} Q/A pairs.")

###  Run Evaluation Loop
We run our RAG system on the test questions and then ask Gemini to act as a judge.

In [ ]:
def evaluate_system(test_set):
    # Check for cache first
    if EVAL_RESULTS_PATH.exists():
        print(f"Loading pre-computed evaluation results from {EVAL_RESULTS_PATH}...")
        with open(EVAL_RESULTS_PATH, 'r') as f:
            return json.load(f)

    if not gemini_model:
        print("No API key and no cached results found.")
        return []

    print("Running RAG evaluation and Expert-Judging via Gemini API...")
    results = []
    for item in test_set:
        # Get RAG answer
        rag_answer = answer_question(item["question"])

        judge_prompt = f"""You are an expert judge evaluating a RAG system.
Compare the 'System Answer' against the 'Gold Answer' for the given 'Question'.

Question: {item['question']}
Gold Answer: {item['answer']}
System Answer: {rag_answer}

Score the System Answer from 1 to 10 based on:
1. Accuracy: Does it match the facts in the Gold Answer?
2. Faithfulness: Does it avoid hallucinating information not in the Gold Answer?
3. Citations: Does it mention Source Document IDs if they were provided?

Return only a JSON object with keys 'score' (int) and 'reasoning' (string)."""

        try:
            eval_response = gemini_model.generate_content(judge_prompt)
            eval_json = json.loads(eval_response.text.strip().replace('```json', '').replace('```', ''))
            results.append({
                "question": item["question"], "gold": item["answer"],
                "rag": rag_answer, "score": eval_json["score"], "reasoning": eval_json["reasoning"]
            })
        except: continue

    # Save results
    with open(EVAL_RESULTS_PATH, 'w') as f:
        json.dump(results, f)
    return results

evaluation_results = evaluate_system(micro_test_set)
if evaluation_results:
    eval_df = pd.DataFrame(evaluation_results)
    print(f"Average System Score: {eval_df['score'].mean():.2f}/10")
    display(eval_df)
else:
    print("No evaluation results to display. Please provide an API key or ensure cache files exist in artifacts/")

### Error Analysis and Improvements

#### Observations
*   [Analyze the eval_df results here. Look for low scores and read the 'reasoning' from the judge.]

#### Potential Improvements
1.  **Re-ranking:** Use a Cross-Encoder to re-rank the top-k results before feeding them to Gemma.
2.  **Context Precision:** Increase the `expand_neighbors` window for very short emails.
3.  **Hybrid Search:** Combine FAISS with keyword-based search (BM25) to handle specific name lookups better.

In [ ]:
#code for testing

# **USE OF GENERATIVE AI**

During the development of this project, a Generative AI assistant (Gemini CLI / Gemini 1.5 Pro) was utilized to assist with coding, debugging, and system optimization. The core architecture and design decisions were made by the students, while the AI was used as an advanced pair-programmer.

Specifically, AI was used for:
1.  **Logic Refinement:** Formulating the dynamic chunking algorithm and transitioning the project from a standard PyTorch/Transformers pipeline to an optimized **GGUF** architecture to solve VRAM constraints.
2.  **Architecture Shift:** Implementing the integration code for `llama-cpp-python` to run the state-of-the-art **Gemma-4-E4B-it-heretic-GGUF** model, which natively handles quantization without needing external libraries like `bitsandbytes`.
3.  **Automated Evaluation:** Building the API calls to `google-generativeai` to construct the 15-question micro-test set and to orchestrate the AI-Judge scoring loop.
4. **Documentation:** Drafting technical markdown summaries and consolidating project dependencies.

**Examples of Prompts Used:**
*   *"How should we fix the token overflow (exceeding 512) due to metadata?"*
*   *"Let's switch to Gemma-4-E4B-it-heretic-GGUF, update the code to use llama-cpp-python and change the documentation to say we don't need bitsandbytes."*
*   *"Can you move all the includes and initialization of the libraries used in LIBRARIES USED block?"*